<a href="https://colab.research.google.com/github/yussef862/blahblahblah/blob/main/Meat_Pipeline_V2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report
print(f' TF: {tf.__version__}')

In [ ]:
data_dir      = '/content/drive/MyDrive/combined_data'
meat_only_dir = '/content/meat_only'
meat_classes  = ['Beef', 'Buffalo meat', 'Camel meat', 'Goat meat', 'Lamb (sheep)']

os.makedirs(meat_only_dir, exist_ok=True)

for cls in meat_classes:
    src = os.path.join(data_dir, cls)
    dst = os.path.join(meat_only_dir, cls)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)

print(' Classes:')
total = 0
for cls in sorted(os.listdir(meat_only_dir)):
    count = len(os.listdir(os.path.join(meat_only_dir, cls)))
    total += count
    print(f'  {cls}: {count} images')
print(f'Total: {total}')

In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

IMG_SIZE = (224, 224)
BATCH    = 16

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.6, 1.4],
    validation_split=0.2
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    meat_only_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    subset='training',
    seed=42
)

val_data = val_gen.flow_from_directory(
    meat_only_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print('Classes:', train_data.class_indices)
print(f'Train: {train_data.samples} | Val: {val_data.samples}')

In [ ]:
base = EfficientNetV2B0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
output = layers.Dense(5, activation='softmax')(x)

model = Model(inputs=base.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(' EfficientNetV2B0 ready')
print(f'Layers: {len(model.layers)}')

In [ ]:
callbacks_p1 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        '/content/drive/MyDrive/ML Project /meat_model_v2b0.keras',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

print(' Phase 1: Training head...')
history_p1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks_p1
)

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        '/content/drive/MyDrive/ML Project /meat_model_v2b0.keras',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

print(' Phase 2: Fine-tuning...')
history_p2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks_p2
)

In [ ]:
val_data.reset()
preds  = model.predict(val_data)
y_pred = np.argmax(preds, axis=1)
y_true = val_data.classes
labels = list(val_data.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

CLASS_NAMES = list(val_data.class_indices.keys())
CONFIDENCE_THRESHOLD = 0.65

def predict(image):
    if image is None:
        return "Please upload an image"

    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img).astype(np.float32)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    confidence = float(np.max(preds))
    predicted = int(np.argmax(preds))

    if confidence < CONFIDENCE_THRESHOLD:
        return "Unknown / Not a meat"

    return f"{CLASS_NAMES[predicted]} ({confidence*100:.1f}%)"

interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(label="Upload meat image"),
    outputs=gr.Textbox(label="Result"),
    title="Meat Classification - V4 Test"
)

interface.queue()
interface.launch(share=True, debug=True)

In [ ]:
def predict_debug(image):
    if image is None:
        return "Please upload an image"

    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img).astype(np.float32)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]

    result = ""
    for i, cls in enumerate(CLASS_NAMES):
        result += f"{cls}: {preds[i]*100:.1f}%\n"

    result += f"\nMax confidence: {float(np.max(preds))*100:.1f}%"
    result += f"\nPredicted: {CLASS_NAMES[int(np.argmax(preds))]}"

    return result

interface2 = gr.Interface(
    fn=predict_debug,
    inputs=gr.Image(label="Upload meat image"),
    outputs=gr.Textbox(label="Debug Result"),
    title="Debug Mode"
)

interface2.queue()
interface2.launch(share=True, debug=True)

In [ ]:
model.save('/content/drive/MyDrive/ML Project /meat_model_v2b0.keras')
print(' Model saved!')

In [ ]:
!pip install transformers -q

import torch
from transformers import CLIPProcessor, CLIPModel
import tensorflow as tf
import numpy as np
from PIL import Image
import gradio as gr
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as preprocess_v4

print('Loading CLIP...')
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print('Loading V3...')
model_v3 = tf.keras.models.load_model('/content/drive/MyDrive/ML Project /best_model_v3.keras')

print('V4 already in memory ')

CLASS_NAMES = ['Beef', 'Buffalo meat', 'Camel meat', 'Goat meat', 'Lamb (sheep)']
CLIP_THRESHOLD = 0.6
CONFIDENCE_THRESHOLD = 0.55

def predict_ensemble(image):
    if image is None:
        return "Please upload an image"

    pil_img = Image.fromarray(image).convert('RGB')

    inputs = clip_processor(
        text=["a photo of raw meat", "a photo that is not meat"],
        images=pil_img,
        return_tensors="pt",
        padding=True
    )
    with torch.no_grad():
        outputs   = clip_model(**inputs)
        probs     = outputs.logits_per_image.softmax(dim=1)
        meat_prob = float(probs[0][0])

    if meat_prob < CLIP_THRESHOLD:
        return "Unknown / Not a meat"

    img_v3 = pil_img.resize((224, 224))
    arr_v3 = np.array(img_v3) / 255.0
    arr_v3 = np.expand_dims(arr_v3, axis=0)
    preds_v3 = model_v3.predict(arr_v3, verbose=0)[0]

    img_v4 = pil_img.resize((224, 224))
    arr_v4 = np.array(img_v4).astype(np.float32)
    arr_v4 = preprocess_v4(arr_v4)
    arr_v4 = np.expand_dims(arr_v4, axis=0)
    preds_v4 = model.predict(arr_v4, verbose=0)[0]

    max_v3 = float(np.max(preds_v3))
    max_v4 = float(np.max(preds_v4))

    if max_v4 >= 0.90:
        final = preds_v4
        source = "V4"
    elif max_v3 >= 0.90:
        final = preds_v3
        source = "V3"
    else:
        final = (0.4 * preds_v3) + (0.6 * preds_v4)
        source = "Ensemble"

    confidence = float(np.max(final))
    predicted  = int(np.argmax(final))

    if confidence < CONFIDENCE_THRESHOLD:
        return "Unknown / Not a meat"

    return f"{CLASS_NAMES[predicted]} ({confidence*100:.1f}%) — {source}"

interface = gr.Interface(
    fn=predict_ensemble,
    inputs=gr.Image(label="Upload meat image"),
    outputs=gr.Textbox(label="Result"),
    title="Meat Classification — CLIP + V3 + V4",
    description="Stage 1: CLIP filters non-meat | Stage 2: V3 + V4 Ensemble"
)

interface.queue()
interface.launch(share=True, debug=True)